# W1-07 Acoustic encoder smoke run (GPU)

Measures peak GPU memory and feature-extraction throughput for the frozen
AraBEST-RQ-300M-6k encoder, and repeats the acceptance checks on CUDA.
See `docs/acoustic_features.md`.

**Before running:** Runtime → Change runtime type → **T4 GPU** (or better) → Save.
Then Runtime → **Run all**.

## 1. Mount Drive and set storage folders
Clips and results go to Drive. The checkpoint (1.24 GB) downloads to Colab's local disk (about 1 minute per session): the Hugging Face cache is unreliable on Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/arabic-dialect-acoustic'
os.environ['DATA_ROOT'] = DATA_ROOT = f'{BASE}/data'
os.environ['OUTPUT_ROOT'] = OUTPUT_ROOT = f'{BASE}/output'
os.environ['CACHE_ROOT'] = '/content/hf-cache'

## 2. Get the code and install the acoustic environment
`SOURCE = 'github'` clones the branch. `SOURCE = 'zip'` uploads `Arabic-Dialect-Identification.zip` built from a local checkout (excluding `.venvs/` and `.cache/`).

In [ ]:
SOURCE = 'github'  # or 'zip'
BRANCH = 'eec-acoustic-encoder'
%cd /content
!rm -rf Arabic-Dialect-Identification Arabic-Dialect-Identification.zip
if SOURCE == 'zip':
    from google.colab import files
    files.upload()  # choose Arabic-Dialect-Identification.zip
    !unzip -q Arabic-Dialect-Identification.zip -d /content
else:
    !git clone -q -b {BRANCH} https://github.com/alizbeeb5/Arabic-Dialect-Identification.git
%cd /content/Arabic-Dialect-Identification
!git log -1 --oneline && git status --short
!pip install -q -r environments/acoustic.txt

## 3. Check the GPU and package versions, then run the unit tests

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, '-c',
    'import torch, speechbrain; print(torch.__version__, speechbrain.__version__, '
    'torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)'],
    capture_output=True, text=True).stdout)
!python -m unittest discover -s tests -p 'test_acoustic_encoder.py'

## 4. Get the 20 pilot clips
Streams about 280 MB. Clip IDs match the official manifest and the local CPU run.

In [ ]:
!python src/data/sample_pilot_clips.py --data-root "$DATA_ROOT"

## 5. Smoke run: acceptance checks, throughput and peak GPU memory

In [ ]:
REV = 'c8da454c9f474bb075d3fe608940ee29e8ccfa43'
MANIFEST = f'{DATA_ROOT}/manifests/adi20_micro/{REV}/pilot-sample/train.jsonl'
AUDIO = f'{DATA_ROOT}/audio/adi20_micro/{REV}'
!python src/models/acoustic_smoke.py --manifest "{MANIFEST}" --audio-root "{AUDIO}" --batch-size 8

## 6. Optional: larger batch and bf16

In [ ]:
!python src/models/acoustic_smoke.py --manifest "{MANIFEST}" --audio-root "{AUDIO}" --batch-size 16
!python src/models/acoustic_smoke.py --manifest "{MANIFEST}" --audio-root "{AUDIO}" --batch-size 8 --precision bf16

## 7. Summary to send back
Copy this output (or download the `run_record.json` files listed).

In [ ]:
import glob, json
for path in sorted(glob.glob(f'{OUTPUT_ROOT}/runs/acoustic_smoke_*/run_record.json')):
    r = json.load(open(path)); m = r['metrics']
    gb = lambda b: None if b is None else round(b / 2**30, 2)
    print(path)
    print(f"  status={r['status']} commit={r['git_commit'][:7]} gpu={r['hardware'].get('gpu')} "
          f"batch={m['batch_size']} precision={m['precision']}")
    print(f"  clips/s={m['clips_per_sec']:.2f} audio-s/s={m['audio_sec_per_sec']:.1f} "
          f"peak_alloc_GB={gb(m['peak_gpu_memory_allocated_bytes'])} "
          f"peak_reserved_GB={gb(m['peak_gpu_memory_reserved_bytes'])}")
    print(f"  repeat_diff={m['repeat_max_abs_diff']:.2e} padding_diff={m['padding_max_abs_diff']:.2e} "
          f"checks={all(m['checks'].values())}")